In [4]:
# pnl_report_embedded.py
# -*- coding: utf-8 -*-
"""
Tek dosya P&L raporu (CSV yok). Veriler aşağıda EMBEDDED_DATA içinde.
Çıktılar:
  - pnl_dashboard.pdf : 5 sayfalık PDF (KPI + grafikler + tablo)
  - output/summary.csv, output/timeline.csv (opsiyonel dışa veri)
Kütüphaneler:
  pip install pandas numpy matplotlib
"""

import os
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ==========================
# 0) Gömülü VERİ (senin gönderdiğin "en güncel" tablo)
# ==========================
# Kolonlar: Ticker, Tarih, Tutar($), İşlem Ücreti, O anki fiyat, İşlem
EMBEDDED_DATA = [
    ("SPGI",  "8.10.2025",  480.61,   0.96,  480.61, "ALIŞ"),
    ("AMT",   "13.10.2025", 369.30,   0.73,  184.65, "ALIŞ"),
    ("BSX",   "13.10.2025", 284.73,   0.56,   94.91, "ALIŞ"),
    ("HD",    "13.10.2025", 378.70,   0.76,  378.70, "ALIŞ"),
    ("PG",    "13.10.2025", 297.90,   0.59,  148.95, "ALIŞ"),
    ("DAL",   "15.10.2025", 122.96,   0.50,   61.48, "ALIŞ"),
    ("DAL",   "20.10.2025", 122.34,   0.50,   61.17, "SATIŞ"),
    ("NVDA",  "15.10.2025", 180.00,   1.50,  179.57, "ALIŞ"),
    ("BRK.B", "15.10.2025", 125.00,   1.50,  492.89, "ALIŞ"),
    ("AAPL",  "16.10.2025", 150.00,   1.50,  247.39, "ALIŞ"),
    ("AAPL",  "20.10.2025", 156.8090869, 1.50, 258.62, "SATIŞ"),
    ("STX",   "22.10.2025", 213.13,   1.50,  213.13, "ALIŞ"),
    ("STX",   "23.10.2025", 225.03,   1.50,  225.03, "SATIŞ"),
    ("META",  "24.10.2025", 366.48,   1.50,  732.95, "ALIŞ"),
    ("VOO",   "24.10.2025", 311.29,   1.50,  622.57, "ALIŞ"),
]

def _parse_date(x):
    for fmt in ("%d.%m.%Y", "%d.%m.%y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(str(x), fmt)
        except Exception:
            pass
    return pd.to_datetime(x, dayfirst=True, errors="coerce")

def load_from_embedded():
    df = pd.DataFrame(EMBEDDED_DATA, columns=["Ticker","Tarih","Amount","Fee","Price","Side"])
    df["Tarih"] = df["Tarih"].apply(_parse_date)
    df["Qty"] = df["Amount"] / df["Price"]  # fraksiyonel lot destekli
    return df.sort_values(["Tarih","Ticker"]).reset_index(drop=True)

def average_cost_pnl(df: pd.DataFrame):
    tickers = sorted(df["Ticker"].unique())
    ledger = {t: {"qty":0.0, "cost":0.0, "realized":0.0, "fees":0.0, "last_px":np.nan} for t in tickers}
    timeline = []

    for _, r in df.iterrows():
        t, dt, amt, fee, px, side, qty = r["Ticker"], r["Tarih"], r["Amount"], r["Fee"], r["Price"], r["Side"], r["Qty"]
        st = ledger[t]
        st["fees"] += fee
        st["last_px"] = px

        if str(side).upper().startswith("ALI"):  # ALIŞ
            st["qty"]  += qty
            st["cost"] += (amt + fee)
        else:  # SATIŞ
            avg_cost = (st["cost"]/st["qty"]) if st["qty"] else 0.0
            proceeds = amt - fee
            cost_out = avg_cost * qty
            st["realized"] += (proceeds - cost_out)
            st["qty"]  -= qty
            st["cost"] -= cost_out
            if abs(st["qty"]) < 1e-10: st["qty"] = 0.0
            if abs(st["cost"]) < 1e-8:  st["cost"] = 0.0

        # her adımda kümülatif snapshot
        unreal = 0.0
        for tk, s in ledger.items():
            if s["qty"] and not pd.isna(s["last_px"]):
                unreal += s["qty"]*s["last_px"] - s["cost"]
        realized = sum(s["realized"] for s in ledger.values())
        timeline.append({"Tarih": dt, "Realized": realized, "Unrealized": unreal, "Total": realized+unreal})

    return ledger, pd.DataFrame(timeline).sort_values("Tarih")

def mark_to_market(ledger: dict):
    rows = []
    for tk, s in ledger.items():
        mv = s["qty"]*s["last_px"] if s["qty"] and s["last_px"]==s["last_px"] else 0.0
        unreal = mv - s["cost"]
        rows.append({
            "Ticker": tk,
            "OpenQty": s["qty"],
            "LastPrice": s["last_px"],
            "MarketValue($)": mv,
            "RemainingCost($)": s["cost"],
            "UnrealizedPnL($)": unreal,
            "RealizedPnL($)": s["realized"],
            "Fees($)": s["fees"],
            "TickerTotalPnL($)": s["realized"] + unreal
        })
    summary = pd.DataFrame(rows).sort_values("Ticker")
    totals = {
        "TotalRealized($)": float(summary["RealizedPnL($)"].sum()),
        "TotalUnrealized($)": float(summary["UnrealizedPnL($)"].sum()),
        "TotalFees($)": float(summary["Fees($)"].sum()),
        "GrandPnL($)": float(summary["TickerTotalPnL($)"].sum())
    }
    return summary, totals

def print_focus(summary: pd.DataFrame, tickers=("BRK.B","VOO")):
    focus = summary[summary["Ticker"].isin(tickers)].copy()
    print("=== AÇIK POZİSYONLAR (BRK.B & VOO) ===" if not focus.empty else "BRK.B ve/veya VOO için açık pozisyon yok.")
    for _, r in focus.iterrows():
        print(f"{r['Ticker']}: Qty={r['OpenQty']:.6f}, Last={r['LastPrice']:.2f}, "
              f"MV={r['MarketValue($)']:.2f}, Cost={r['RemainingCost($)']:.2f}, "
              f"Unreal={r['UnrealizedPnL($)']:.2f}")

def make_pdf(df: pd.DataFrame, out_pdf="pnl_dashboard.pdf", export_csv=True):
    # hesapla
    ledger, timeline = average_cost_pnl(df)
    summary, totals = mark_to_market(ledger)

    # çıktı klasörü
    os.makedirs("output", exist_ok=True)
    if export_csv:
        summary.to_csv("output/summary.csv", index=False, encoding="utf-8")
        timeline.to_csv("output/timeline.csv", index=False, encoding="utf-8")

    # PDF
    with PdfPages(out_pdf) as pdf:
        # Sayfa 1 — KPI
        fig = plt.figure(figsize=(8.27, 11.69))  # A4 dikey
        plt.axis("off")
        lines = [
            "Trading P&L Dashboard",
            "",
            f"Genel P&L: {totals['GrandPnL($)']:.2f} USD",
            f"Gerçekleşen: {totals['TotalRealized($)']:.2f} USD",
            f"Gerçekleşmemiş: {totals['TotalUnrealized($)']:.2f} USD",
            f"Toplam Ücret: {totals['TotalFees($)']:.2f} USD",
        ]
        plt.text(0.1, 0.9, "\n".join(lines), fontsize=16, va="top")
        pdf.savefig(fig); plt.close(fig)

        # Sayfa 2 — Toplam P&L (zaman)
        fig = plt.figure()
        plt.plot(timeline["Tarih"], timeline["Total"])
        plt.title("Toplam P&L (Zaman)"); plt.xlabel("Tarih"); plt.ylabel("Toplam P&L ($)")
        pdf.savefig(fig); plt.close(fig)

        # Sayfa 3 — Realized vs Unrealized
        fig = plt.figure()
        plt.plot(timeline["Tarih"], timeline["Realized"], label="Gerçekleşen")
        plt.plot(timeline["Tarih"], timeline["Unrealized"], label="Gerçekleşmemiş")
        plt.title("Gerçekleşen vs Gerçekleşmemiş P&L"); plt.xlabel("Tarih"); plt.ylabel("P&L ($)")
        plt.legend()
        pdf.savefig(fig); plt.close(fig)

        # Sayfa 4 — Ticker bazında toplam P&L
        fig = plt.figure()
        plt.bar(summary["Ticker"], summary["TickerTotalPnL($)"])
        plt.title("Ticker Bazında Toplam P&L"); plt.xlabel("Ticker"); plt.ylabel("Toplam P&L ($)")
        pdf.savefig(fig); plt.close(fig)

        # Sayfa 5 — Detay Tablo
        fig = plt.figure(figsize=(11.69, 8.27))  # A4 yatay
        plt.axis("off")
        tbl = plt.table(cellText=summary.round(2).values, colLabels=summary.columns, loc='center')
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1,1.4)
        plt.title("Detay Tablo (Summary)")
        pdf.savefig(fig); plt.close(fig)

    # Konsol özeti
    print("\n=== GENEL SONUÇLAR ===")
    print(f"Gerçekleşen P&L:     {totals['TotalRealized($)']:.2f} USD")
    print(f"Gerçekleşmemiş P&L:  {totals['TotalUnrealized($)']:.2f} USD")
    print(f"Toplam Ücret:        {totals['TotalFees($)']:.2f} USD")
    print(f"GENEL P&L:           {totals['GrandPnL($)']:.2f} USD\n")
    print_focus(summary, ("BRK.B","VOO"))
    print(f"\nPDF oluşturuldu: {out_pdf}")
    if export_csv:
        print("CSV çıktıları: output/summary.csv, output/timeline.csv")

if __name__ == "__main__":
    df = load_from_embedded()
    make_pdf(df, out_pdf="pnl_dashboard.pdf", export_csv=True)



=== GENEL SONUÇLAR ===
Gerçekleşen P&L:     11.09 USD
Gerçekleşmemiş P&L:  -9.60 USD
Toplam Ücret:        16.60 USD
GENEL P&L:           1.49 USD

=== AÇIK POZİSYONLAR (BRK.B & VOO) ===
BRK.B: Qty=0.253606, Last=492.89, MV=125.00, Cost=126.50, Unreal=-1.50
VOO: Qty=0.500008, Last=622.57, MV=311.29, Cost=312.79, Unreal=-1.50

PDF oluşturuldu: pnl_dashboard.pdf
CSV çıktıları: output/summary.csv, output/timeline.csv


In [6]:
# trade_charts_daily.py
# -*- coding: utf-8 -*-
"""
Her ticker için son 1 ayın GÜNLÜK fiyatlarını yfinance'tan çeker,
ALIŞ/SATIŞ noktalarını grafiğe koyar ve sadece İŞLEM FİYATINI (adet yok) yazdırır.
PDF ÜRETMEZ. Her ticker için tek bir PNG dosyası kaydeder.

Kurulum:
    pip install pandas numpy matplotlib yfinance
Çalıştırma:
    python trade_charts_daily.py
"""

import os
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- Gömülü işlemler (senin son paylaştığın tablo) ----
# Kolonlar: Ticker, Tarih, Tutar($), Ücret, Fiyat, İşlem
TRADES = [
    ("SPGI",  "8.10.2025",   480.61,     0.96,  480.61, "ALIŞ"),
    ("AMT",   "13.10.2025",  369.30,     0.73,  184.65, "ALIŞ"),
    ("BSX",   "13.10.2025",  284.73,     0.56,   94.91, "ALIŞ"),
    ("HD",    "13.10.2025",  378.70,     0.76,  378.70, "ALIŞ"),
    ("PG",    "13.10.2025",  297.90,     0.59,  148.95, "ALIŞ"),
    ("DAL",   "15.10.2025",  122.96,     0.50,   61.48, "ALIŞ"),
    ("DAL",   "20.10.2025",  122.34,     0.50,   61.17, "SATIŞ"),
    ("NVDA",  "15.10.2025",  180.00,     1.50,  179.57, "ALIŞ"),
    ("BRK.B", "15.10.2025",  125.00,     1.50,  492.89, "ALIŞ"),
    ("AAPL",  "16.10.2025",  150.00,     1.50,  247.39, "ALIŞ"),
    ("AAPL",  "20.10.2025",  156.8090869,1.50,  258.62, "SATIŞ"),
    ("STX",   "22.10.2025",  213.13,     1.50,  213.13, "ALIŞ"),
    ("STX",   "23.10.2025",  225.03,     1.50,  225.03, "SATIŞ"),
    ("META",  "24.10.2025",  366.48,     1.50,  732.95, "ALIŞ"),
    ("VOO",   "24.10.2025",  311.29,     1.50,  622.57, "ALIŞ"),
]

# Yahoo Finance sembol eşleştirmesi (örn. BRK.B -> BRK-B)
YF_MAPPING = {
    "BRK.B": "BRK-B",
}

def parse_date_tr(x: str) -> datetime:
    for fmt in ("%d.%m.%Y", "%d.%m.%y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(str(x), fmt)
        except Exception:
            pass
    # yine de parse edilemezse NaT dönebilir
    return pd.to_datetime(x, dayfirst=True, errors="coerce")

def prepare_trades(trades):
    df = pd.DataFrame(trades, columns=["Ticker","Tarih","Amount","Fee","TradePrice","Side"])
    df["Date"] = df["Tarih"].apply(parse_date_tr)
    # sadece ihtiyacımız olan kolonlar
    df = df[["Ticker","Date","TradePrice","Side"]].sort_values(["Ticker","Date"]).reset_index(drop=True)
    return df

def fetch_daily_prices(ticker: str, start: datetime, end: datetime) -> pd.Series:
    """yfinance'tan günlük kapanışlar. Başarısız olursa boş döner."""
    try:
        import yfinance as yf
        yf_ticker = YF_MAPPING.get(ticker, ticker)
        data = yf.download(yf_ticker, start=start, end=end, interval="1d", progress=False, threads=True)
        if data is None or len(data) == 0:
            return pd.Series(dtype=float)
        close = data["Close"].copy()
        close.name = "Close"
        return close
    except Exception:
        return pd.Series(dtype=float)

def plot_ticker_daily(ax, close: pd.Series, trades_tkr: pd.DataFrame, ticker: str):
    # Fiyat çizgisi
    ax.plot(close.index, close.values, linewidth=1.5)
    ax.set_title(f"{ticker} — Son 1 Ay Günlük Fiyat")
    ax.set_xlabel("Tarih"); ax.set_ylabel("Fiyat")
    ax.grid(True, linestyle="--", alpha=0.3)

    # ALIŞ / SATIŞ noktaları (sadece fiyat etiketi)
    buys  = trades_tkr[trades_tkr["Side"].str.upper().str.startswith("ALI")]
    sells = trades_tkr[~trades_tkr["Side"].str.upper().str.startswith("ALI")]

    # ALIŞ: ▲ ve fiyat etiketi
    if not buys.empty:
        ax.scatter(buys["Date"], buys["TradePrice"], marker="^", s=70, label="ALIŞ")
        for _, r in buys.iterrows():
            ax.annotate(f"{r['TradePrice']:.2f}", (r["Date"], r["TradePrice"]),
                        textcoords="offset points", xytext=(0,8), ha="center", fontsize=8)

    # SATIŞ: ▼ ve fiyat etiketi
    if not sells.empty:
        ax.scatter(sells["Date"], sells["TradePrice"], marker="v", s=70, label="SATIŞ")
        for _, r in sells.iterrows():
            ax.annotate(f"{r['TradePrice']:.2f}", (r["Date"], r["TradePrice"]),
                        textcoords="offset points", xytext=(0,-12), ha="center", fontsize=8)

    if (not buys.empty) or (not sells.empty):
        ax.legend(loc="best")

def main():
    os.makedirs("charts", exist_ok=True)

    trades = prepare_trades(TRADES)
    tickers = trades["Ticker"].unique()

    # Pencere: son işlemin tarihine göre 1 ay (yoksa bugüne göre)
    max_trade_dt = trades["Date"].max()
    base_end = max_trade_dt if pd.notna(max_trade_dt) else datetime.utcnow()
    start = (base_end - timedelta(days=31)).replace(hour=0, minute=0, second=0, microsecond=0)
    end   = (base_end + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)

    for tkr in tickers:
        tdf = trades[trades["Ticker"] == tkr].copy()

        # Günlük kapanışları çek
        close = fetch_daily_prices(tkr, start, end)

        # Eğer veri gelmezse, en azından trade noktalarını çizmek için bir fallback çizgi yapalım:
        # (kullanıcı "gün gün verileri çek" dedi, o yüzden fallback kullanmamak daha doğru;
        #  yine de boş kalmasın diye basit trade-bazlı çizgi ekliyoruz)
        if close.empty:
            # Trade günleri için seri, diğer günler NaN; ffill ile basit çizgi
            idx = pd.date_range(start=start, end=end, freq="D")
            s = pd.Series(index=pd.to_datetime(tdf["Date"]), data=tdf["TradePrice"].values).sort_index()
            s = s[~s.index.duplicated(keep="last")]
            close = s.reindex(idx).ffill()
            close.name = "Close"

        # Çiz ve kaydet
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_ticker_daily(ax, close, tdf, tkr)
        plt.tight_layout()
        out_png = os.path.join("charts", f"{tkr}.png")
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

    print("Tamamdır. Grafikler charts/ klasörüne kaydedildi (her ticker için 1 PNG).")

if __name__ == "__main__":
    main()


Tamamdır. Grafikler charts/ klasörüne kaydedildi (her ticker için 1 PNG).
